In [1]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("raw_brasileirao") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO + S3 configurados!")

Spark version: 3.5.0
Delta + MinIO + S3 configurados!


In [2]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
LEAGUE = "brasileirao-serie-a"
SEASON = "2026"
ENVIRONMENT = "dev"
REPROC_MODE = False

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"

LANDING_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/landing/{LEAGUE}"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Landing: {LANDING_PATH}")
print(f"Raw:     {RAW_PATH}")

League: brasileirao-serie-a | Season: 2026 | Env: dev | Reproc: False
Landing: s3a://eng-prd-data-lake/dev/landing/brasileirao-serie-a
Raw:     s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a


In [3]:
from pyspark.sql.functions import current_timestamp, lit, explode, col
import json

CHECKPOINT_KEY = f"{ENVIRONMENT}/raw/{LEAGUE}/_processed_files.json"

def get_processed_files():
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=CHECKPOINT_KEY)
        return set(json.loads(obj['Body'].read()))
    except:
        return set()

def save_processed_files(files):
    s3.put_object(
        Bucket=BUCKET,
        Key=CHECKPOINT_KEY,
        Body=json.dumps(list(files)),
        ContentType="application/json"
    )

def list_landing_files(data_type):
    prefix = f"{ENVIRONMENT}/landing/{LEAGUE}/season={SEASON}/{data_type}/"
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]

def ingest_to_raw(data_type):
    raw = f"{RAW_PATH}/{data_type}"
    processed = get_processed_files()
    new_files = [f for f in list_landing_files(data_type) if f not in processed]

    if not new_files:
        print(f"[SKIP] {data_type} — nenhum arquivo novo.")
        return

    print(f"[INFO] {data_type} — {len(new_files)} arquivo(s) novo(s).")

    for file_key in new_files:
        path = f"s3a://{BUCKET}/{file_key}"
        df = spark.read.option("multiline", "true").json(path)
        df = df.withColumn("record", explode(col("data"))) \
               .drop("data") \
               .withColumn("ingested_at", current_timestamp()) \
               .withColumn("league", lit(LEAGUE)) \
               .withColumn("season", lit(SEASON))

        df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .partitionBy("season") \
            .save(raw)

        processed.add(file_key)
        save_processed_files(processed)
        print(f"[OK] {file_key} processado!")

    print(f"[OK] {data_type} finalizado!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    ingest_to_raw(data_type)

print("Camada Raw finalizada!")

[SKIP] standing — nenhum arquivo novo.
[SKIP] rounds — nenhum arquivo novo.
[SKIP] events — nenhum arquivo novo.
[SKIP] players — nenhum arquivo novo.
[SKIP] team_stats — nenhum arquivo novo.
[INFO] player_stats — 1 arquivo(s) novo(s).
[OK] dev/landing/brasileirao-serie-a/season=2026/player_stats/extracted_at=2026-03-08/player_stats.json processado!
[OK] player_stats finalizado!

Camada Raw finalizada!
